In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from langchain_core.documents import Document

# Load medicine datasets
df1 = pd.read_csv("data/A_Z_medicines_dataset_of_india.csv")
df2 = pd.read_csv("data/medDataset_processed.csv")

df1_clean = df1[['name','price(₹)','manufacturer_name','type',
                  'pack_size_label','short_composition1',
                  'short_composition2']].copy()
df1_clean.fillna("Not available", inplace=True)
df1_clean['text'] = df1_clean.apply(lambda row:
    f"Medicine: {row['name']}\n"
    f"Price: ₹{row['price(₹)']}\n"
    f"Manufacturer: {row['manufacturer_name']}\n"
    f"Type: {row['type']}\n"
    f"Pack Size: {row['pack_size_label']}\n"
    f"Composition: {row['short_composition1']}, {row['short_composition2']}",
    axis=1)

df2_clean = df2[['Question','Answer']].dropna()
df2_clean['text'] = df2_clean.apply(lambda row:
    f"Question: {row['Question']}\nAnswer: {row['Answer']}", axis=1)

# Load medical stores dataset
stores_df = pd.read_excel("data/jaipur_medical_stores_ml.xlsx")
stores_df['Latitude'] = pd.to_numeric(stores_df['Latitude'], errors='coerce')
stores_df['Longitude'] = pd.to_numeric(stores_df['Longitude'], errors='coerce')
stores_df = stores_df.dropna(subset=['Latitude', 'Longitude'])

print(f"✅ Medicines loaded: {len(df1_clean)}")
print(f"✅ Q&A loaded: {len(df2_clean)}")
print(f"✅ Medical stores loaded: {len(stores_df)}")
print(f"📍 Localities covered: {stores_df['Locality'].nunique()}")

✅ Medicines loaded: 253973
✅ Q&A loaded: 16407
✅ Medical stores loaded: 383
📍 Localities covered: 40


In [2]:
# Use updated langchain-huggingface and langchain-chroma packages (with fallback)
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

try:
    from langchain_chroma import Chroma
except ImportError:
    from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    persist_directory="vectorstore",
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print(f"✅ Vectorstore loaded!")
print(f"📊 Total documents: {vectorstore._collection.count()}")

# Quick test
results = vectorstore.similarity_search("paracetamol", k=2)
print(f"\n🧪 Search test:")
for r in results:
    print(f"→ {r.page_content[:80]}")


✅ Vectorstore loaded!
📊 Total documents: 609516

🧪 Search test:
→ Medicine: Paragreat 250mg Suspension
Price: ₹44.35
Manufacturer: Mankind Pharma 
→ Medicine: Paragreat 250mg Suspension
Price: ₹44.35
Manufacturer: Mankind Pharma 


In [3]:
import requests
import json
from geopy.distance import geodesic

OLLAMA_BASE = "http://localhost:11434"

def get_available_model():
    """Auto-detect which model works with the running Ollama version."""
    try:
        resp = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
        models = [m["name"] for m in resp.json().get("models", [])]
        print(f"Available models: {models}")
    except Exception as e:
        print(f"Cannot reach Ollama: {e}")
        return None, None

    for model in models:
        try:
            r = requests.post(f"{OLLAMA_BASE}/api/chat",
                json={"model": model, "messages": [{"role": "user", "content": "hi"}], "stream": False},
                timeout=30)
            if r.status_code == 200:
                print(f"Model '{model}' works with /api/chat")
                return model, "chat"
        except Exception:
            pass
        try:
            r = requests.post(f"{OLLAMA_BASE}/api/generate",
                json={"model": model, "prompt": "hi", "stream": False},
                timeout=30)
            if r.status_code == 200:
                print(f"Model '{model}' works with /api/generate")
                return model, "generate"
        except Exception:
            pass
    return None, None

ACTIVE_MODEL, API_MODE = get_available_model()

# ── Streaming invoke ────────────────────────────────────────────────────────
def ollama_stream(prompt: str):
    """Generator: yields text chunks as Ollama streams them."""
    if ACTIVE_MODEL is None:
        yield "Ollama is not available. Please run: ollama serve"
        return
    try:
        if API_MODE == "chat":
            with requests.post(
                f"{OLLAMA_BASE}/api/chat",
                json={
                    "model": ACTIVE_MODEL,
                    "messages": [{"role": "user", "content": prompt}],
                    "stream": True,
                    "options": {"num_predict": 400, "temperature": 0.3}
                },
                stream=True, timeout=120
            ) as resp:
                for line in resp.iter_lines():
                    if line:
                        chunk = json.loads(line)
                        token = chunk.get("message", {}).get("content", "")
                        if token:
                            yield token
                        if chunk.get("done"):
                            break
        else:
            with requests.post(
                f"{OLLAMA_BASE}/api/generate",
                json={
                    "model": ACTIVE_MODEL,
                    "prompt": prompt,
                    "stream": True,
                    "options": {"num_predict": 400, "temperature": 0.3}
                },
                stream=True, timeout=120
            ) as resp:
                for line in resp.iter_lines():
                    if line:
                        chunk = json.loads(line)
                        token = chunk.get("response", "")
                        if token:
                            yield token
                        if chunk.get("done"):
                            break
    except Exception as e:
        yield f"LLM error: {e}"

# ── Store finder ────────────────────────────────────────────────────────────
def find_nearest_stores(user_area=None, user_lat=None, user_lon=None, top_n=3):
    df = stores_df.copy()
    if user_lat and user_lon:
        def calc_dist(row):
            return geodesic((user_lat, user_lon), (row["Latitude"], row["Longitude"])).km
        df["distance_km"] = df.apply(calc_dist, axis=1)
        df = df.sort_values("distance_km")
    elif user_area:
        area_lower = user_area.lower()
        df["score"] = (
            df["Locality"].str.lower().str.contains(area_lower, na=False).astype(int) +
            df["Address"].str.lower().str.contains(area_lower, na=False).astype(int) +
            df["Zone"].str.lower().str.contains(area_lower, na=False).astype(int)
        )
        df = df[df["score"] > 0].sort_values("score", ascending=False)
        df["distance_km"] = 0
    results = []
    for _, row in df.head(top_n).iterrows():
        phone = str(row.get("Phone Primary", "N/A"))
        if phone.startswith("91") and len(phone) >= 12:
            phone = phone[2:]
        has_delivery = row.get("Has Delivery", False)
        delivery_radius = row.get("Delivery Radius Km", None)
        import pandas as pd
        delivery = (f"Delivery: {delivery_radius}km radius"
                    if pd.notna(delivery_radius) else "Delivery available") \
                    if has_delivery == True else "No delivery"
        timing = "Open 24x7" if row.get("Is 24X7", False) == True else \
                 f"{row.get('Opening Time','N/A')} - {row.get('Closing Time','N/A')}"
        dist = row.get("distance_km", 0)
        dist_text = f"{dist:.1f}km away" if dist > 0 else row.get("Locality", "N/A")
        rating  = row.get("Google Rating", "N/A")
        reviews = int(row.get("Review Count", 0))
        results.append(
            f"{row['Store Name']} ({dist_text})\n"
            f"  Rating: {rating}/5 ({reviews} reviews)\n"
            f"  Hours: {timing}\n"
            f"  {delivery}\n"
            f"  Phone: {phone}\n"
            f"  {row.get('Address','N/A')}\n"
            f"  Maps: {row.get('Google Maps Link','N/A')}"
        )
    return results

# ── Main Q&A (returns generator for streaming) ───────────────────────────────
def ask_with_location(question, area=None):
    docs = retriever.invoke(question)
    # Truncate each doc to 200 chars — keeps context tight and LLM fast
    context = "\n\n".join([d.page_content[:200] for d in docs])

    store_section = ""
    if area and area.strip():
        stores = find_nearest_stores(user_area=area, top_n=3)
        if stores:
            store_section = "\n\nNEARBY STORES IN " + area.upper() + ":\n"
            store_section += "\n\n".join(stores)
        else:
            store_section = f"\n\nNo stores found in {area}. Try: Malviya Nagar, Vaishali Nagar."

    prompt = (
        "You are MediBot Jaipur. Answer concisely about Indian medicines.\n"
        "List medicine names, prices, compositions. Mention nearby stores.\n"
        "Always remind to consult a doctor. Be brief and clear.\n\n"
        f"MEDICINE CONTEXT:\n{context}\n"
        f"{store_section}\n\n"
        f"Question: {question}\n"
        f"Area: {area or 'Not specified'}\n\n"
        "Answer:"
    )
    return ollama_stream(prompt)  # returns a generator

print("All functions ready!")
print("\nTesting store search...")
test_stores = find_nearest_stores(user_area="Malviya Nagar", top_n=2)
for s in test_stores:
    print(s)
    print()


Available models: ['gemma4:latest', 'mistral:latest', 'llama3:latest']
Model 'gemma4:latest' works with /api/chat
All functions ready!

Testing store search...
Apollo Pharmacy (Malviya Nagar)
  Rating: 3.4/5 (302 reviews)
  Hours: 07:00 - 23:00
  No delivery
  Phone: 9848870700
  Shop No. 243, Malviya Nagar Colony, Malviya Nagar, Jaipur - 302004
  Maps: https://maps.google.com/?q=26.862531,75.811106

City Med House (Malviya Nagar)
  Rating: 4.7/5 (2790 reviews)
  Hours: 07:00 - 21:00
  Delivery: 5.0km radius
  Phone: 8163907779
  Shop No. 44, Malviya Nagar Circle, Malviya Nagar, Jaipur - 302018
  Maps: https://maps.google.com/?q=26.861947,75.803862



In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# MediBot Jaipur — Complete Gradio App
# Run: just execute this cell in Jupyter Notebook
# Replace ask_with_location() with your actual RAG streaming function
# ─────────────────────────────────────────────────────────────────────────────

import gradio as gr
import time

# ══════════════════════════════════════════════════════════════════════════════
# YOUR RAG FUNCTION — replace this with your actual implementation
# Must be a generator that yields string tokens
# ══════════════════════════════════════════════════════════════════════════════
def ask_with_location(question, area=""):
    """
    Replace this placeholder with your actual RAG pipeline.
    Your real function should:
      - query ChromaDB for relevant docs
      - call Mistral via Ollama
      - yield response tokens one by one
    """
    mock = f"This is a placeholder response for: '{question}'"
    if area.strip():
        mock += f" (searching near {area})"
    mock += "\n\nReplace ask_with_location() with your actual RAG streaming function."
    for word in mock.split():
        yield word + " "
        time.sleep(0.04)


# ══════════════════════════════════════════════════════════════════════════════
# GRADIO LOGIC
# ══════════════════════════════════════════════════════════════════════════════
def ask_assistant(question, area, history):
    area = area or ""
    if not question or not question.strip():
        return history, ""
    display_q = f"[{area}]  {question}" if area.strip() else question
    history = history + [
        {"role": "user",      "content": display_q},
        {"role": "assistant", "content": ""},
    ]
    stream = ask_with_location(question, area=area)
    accumulated = ""
    for token in stream:
        accumulated += token
        history[-1]["content"] = accumulated
        yield history, ""

def clear_chat():
    return [], ""


# ══════════════════════════════════════════════════════════════════════════════
# CSS
# ══════════════════════════════════════════════════════════════════════════════
css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800;900&family=Sora:wght@600;700;800&display=swap');

*, *::before, *::after { box-sizing: border-box; }

footer, .show-api, .built-with { display: none !important; }

.gradio-container {
    font-family: 'Inter', sans-serif !important;
    background: #0d1117 !important;
    max-width: 1400px !important;
    margin: 0 auto !important;
    padding: 0 !important;
    min-height: 100vh !important;
}

/* ── HEADER ── */
.mb-hdr {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 18px 32px;
    background: #0d1117;
    border-bottom: 1px solid #1e2d45;
}

.mb-brand { display: flex; align-items: center; gap: 14px; }

.mb-icon {
    width: 46px; height: 46px;
    background: linear-gradient(135deg, #2563eb, #0891b2);
    border-radius: 12px;
    display: flex; align-items: center; justify-content: center;
    font-size: 1.4em;
    flex-shrink: 0;
}

.mb-name {
    font-family: 'Sora', sans-serif;
    font-size: 1.15em;
    font-weight: 800;
    color: #f1f5f9;
    letter-spacing: -0.3px;
}

.mb-tagline {
    font-size: 0.7em;
    color: #475569;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.8px;
    margin-top: 2px;
}

.mb-disc {
    font-size: 0.74em;
    color: #475569;
    font-weight: 500;
    text-align: center;
}

.mb-status {
    display: flex;
    align-items: center;
    gap: 8px;
    padding: 8px 16px;
    border: 1px solid rgba(34, 197, 94, 0.35);
    border-radius: 20px;
    background: rgba(34, 197, 94, 0.07);
    font-size: 0.77em;
    color: #86efac;
    font-weight: 700;
    letter-spacing: 0.5px;
}

.mb-dot {
    width: 7px; height: 7px;
    background: #22c55e;
    border-radius: 50%;
    animation: pulse 2s ease-in-out infinite;
}

@keyframes pulse { 0%, 100% { opacity: 1; } 50% { opacity: 0.35; } }
@keyframes slideUp { from { opacity: 0; transform: translateY(16px); } to { opacity: 1; transform: translateY(0); } }

/* ── STATS BAR ── */
.mb-stats {
    display: flex !important;
    gap: 0 !important;
    border-bottom: 1px solid #1e2d45 !important;
    margin: 0 !important;
    padding: 0 !important;
    width: 100%;
}

.mb-sc {
    flex: 1;
    padding: 14px 20px;
    text-align: center;
    border-right: 1px solid #1e2d45;
    background: #0d1117;
    transition: background 0.2s;
    cursor: default;
}

.mb-sc:last-child { border-right: none; }
.mb-sc:hover { background: #0b1522; }

.mb-sv {
    font-family: 'Sora', sans-serif;
    font-size: 1.4em;
    font-weight: 800;
    color: #3b82f6;
    letter-spacing: -0.5px;
    line-height: 1;
}

.mb-sv.g { color: #22c55e; }
.mb-sv.o { color: #f59e0b; }
.mb-sv.w { color: #e2e8f0; }

.mb-sl {
    font-size: 0.62em;
    color: #475569;
    text-transform: uppercase;
    letter-spacing: 1px;
    font-weight: 700;
    margin-top: 4px;
}

/* ── REMOVE GRADIO DEFAULT PADDING ── */
.gradio-container > .wrap { background: transparent !important; padding: 0 !important; }
.gradio-container .gap { gap: 0 !important; }
.gradio-container .column { padding: 0 !important; }

/* ── CHATBOT ── */
#mb-bot {
    border-radius: 0 !important;
    border: none !important;
    background: #0d1117 !important;
    min-height: 560px !important;
    max-height: 560px !important;
    animation: slideUp 0.5s ease;
}

#mb-bot .message-wrap {
    padding: 20px 24px !important;
    gap: 14px !important;
    background: #0d1117 !important;
}

#mb-bot .message.user {
    justify-content: flex-end !important;
}

#mb-bot .message.user > div {
    background: #1e3a5f !important;
    border: 1px solid #2d5a8a !important;
    color: #e0e7ff !important;
    border-radius: 16px 16px 4px 16px !important;
    font-size: 0.9em !important;
    line-height: 1.7 !important;
    max-width: 75% !important;
    padding: 12px 16px !important;
    font-weight: 500;
    box-shadow: none !important;
}

#mb-bot .message.bot > div {
    background: #131d2e !important;
    border: 1px solid #1e2d45 !important;
    color: #e2e8f0 !important;
    border-radius: 16px 16px 16px 4px !important;
    font-size: 0.9em !important;
    line-height: 1.75 !important;
    max-width: 82% !important;
    padding: 12px 16px !important;
    box-shadow: none !important;
}

/* ── INPUTS ── */
#mb-loc, #mb-q {
    animation: slideUp 0.5s ease 0.1s both;
    background: transparent !important;
}

#mb-loc label span, #mb-q label span {
    font-size: 0.65em !important;
    font-weight: 800 !important;
    color: #475569 !important;
    text-transform: uppercase !important;
    letter-spacing: 1.4px !important;
}

#mb-loc textarea, #mb-loc input,
#mb-q textarea, #mb-q input {
    background: #0b1522 !important;
    border: 1px solid #1e2d45 !important;
    border-radius: 10px !important;
    color: #e2e8f0 !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 0.88em !important;
    padding: 10px 14px !important;
    transition: border-color 0.2s, box-shadow 0.2s !important;
    box-shadow: none !important;
}

#mb-loc textarea:focus, #mb-loc input:focus,
#mb-q textarea:focus, #mb-q input:focus {
    border-color: #2563eb !important;
    box-shadow: 0 0 0 3px rgba(37, 99, 235, 0.15) !important;
    outline: none !important;
}

#mb-loc textarea::placeholder, #mb-loc input::placeholder,
#mb-q textarea::placeholder, #mb-q input::placeholder {
    color: #334155 !important;
}

/* ── SEND BUTTON ── */
#mb-send button {
    background: linear-gradient(135deg, #2563eb, #0891b2) !important;
    border: none !important;
    border-radius: 10px !important;
    color: #fff !important;
    font-family: 'Sora', sans-serif !important;
    font-size: 0.88em !important;
    font-weight: 700 !important;
    padding: 10px 24px !important;
    transition: transform 0.15s, box-shadow 0.15s !important;
    cursor: pointer;
    height: 100% !important;
    min-height: 44px !important;
    letter-spacing: 0.4px;
    box-shadow: none !important;
}

#mb-send button:hover {
    transform: translateY(-1px) !important;
    box-shadow: 0 8px 24px rgba(37, 99, 235, 0.4) !important;
    background: linear-gradient(135deg, #1d4ed8, #0284c7) !important;
}

#mb-send button:active {
    transform: translateY(0) scale(0.98) !important;
}

/* ── CLEAR BUTTON ── */
#mb-clear button {
    background: transparent !important;
    border: 1px solid #1e2d45 !important;
    border-radius: 8px !important;
    color: #475569 !important;
    font-size: 0.72em !important;
    font-weight: 700 !important;
    padding: 7px 16px !important;
    text-transform: uppercase;
    letter-spacing: 0.8px;
    transition: all 0.2s !important;
    box-shadow: none !important;
}

#mb-clear button:hover {
    border-color: rgba(239, 68, 68, 0.5) !important;
    color: #f87171 !important;
    background: rgba(239, 68, 68, 0.07) !important;
}

/* ── QUICK BUTTONS ── */
#mb-q1 button, #mb-q2 button, #mb-q3 button,
#mb-q4 button, #mb-q5 button, #mb-q6 button {
    background: #0d1117 !important;
    border: 1px solid #1a2740 !important;
    border-radius: 8px !important;
    color: #94a3b8 !important;
    font-size: 0.8em !important;
    font-weight: 500 !important;
    width: 100% !important;
    text-align: left !important;
    justify-content: flex-start !important;
    padding: 10px 12px !important;
    line-height: 1.5 !important;
    margin-bottom: 7px !important;
    transition: all 0.2s !important;
    cursor: pointer;
    box-shadow: none !important;
}

#mb-q1 button:hover, #mb-q2 button:hover, #mb-q3 button:hover,
#mb-q4 button:hover, #mb-q5 button:hover, #mb-q6 button:hover {
    background: #131d2e !important;
    border-color: #2d4a72 !important;
    color: #c7d8f0 !important;
    transform: translateX(4px) !important;
    box-shadow: none !important;
}

/* ── SIDEBAR PANEL ── */
#mb-panel {
    background: #0b1119;
    border-left: 1px solid #1e2d45;
    padding: 20px 18px;
    animation: slideUp 0.5s ease 0.15s both;
    height: 100%;
}

.mb-sh {
    font-size: 0.63em;
    font-weight: 800;
    color: #334155;
    text-transform: uppercase;
    letter-spacing: 2px;
    padding-bottom: 12px;
    border-bottom: 1px solid #131d2e;
    margin-bottom: 12px;
}

.mb-sd {
    height: 1px;
    background: #131d2e;
    margin: 14px 0;
}

.mb-ss {
    font-size: 0.63em;
    font-weight: 800;
    color: #334155;
    text-transform: uppercase;
    letter-spacing: 2px;
    margin: 0 0 10px 0;
}

.mb-tags {
    display: flex;
    flex-wrap: wrap;
    gap: 6px;
}

.mb-tag {
    padding: 5px 10px;
    background: #0d1117;
    border: 1px solid #1a2740;
    border-radius: 6px;
    font-size: 0.71em;
    color: #475569;
    font-weight: 600;
    transition: all 0.2s;
    cursor: default;
    letter-spacing: 0.3px;
}

.mb-tag:hover {
    border-color: #2d5a8a;
    color: #93c5fd;
    background: #0e1e35;
}

.mb-tr {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 8px 0;
    border-bottom: 1px solid #0f1923;
    font-size: 0.8em;
}

.mb-tr:last-child { border-bottom: none; }
.mb-tk { color: #334155; font-weight: 600; }
.mb-tv { color: #34d399; font-weight: 700; }

/* ── SCROLLBAR ── */
::-webkit-scrollbar { width: 5px; height: 5px; }
::-webkit-scrollbar-track { background: transparent; }
::-webkit-scrollbar-thumb { background: #1e2d45; border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: #2d4a72; }

/* ── RESPONSIVE ── */
@media (max-width: 768px) {
    .mb-hdr { flex-direction: column; gap: 12px; padding: 16px 20px; }
    .mb-stats { flex-wrap: wrap; }
    .mb-sc { min-width: 50%; border-bottom: 1px solid #1e2d45; }
}
"""


# ══════════════════════════════════════════════════════════════════════════════
# HTML BLOCKS
# ══════════════════════════════════════════════════════════════════════════════
FONTS_HTML = """
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800;900&family=Sora:wght@600;700;800&display=swap" rel="stylesheet">
"""

HDR_HTML = """
<div class="mb-hdr">
  <div class="mb-brand">
    <div class="mb-icon">⚕</div>
    <div class="mb-info">
      <div class="mb-name">MediBot Jaipur</div>
      <div class="mb-tagline">Pharmacy Companion · RAG-powered</div>
    </div>
  </div>
  <div class="mb-disc">Not a substitute for professional medical advice</div>
  <div class="mb-status"><div class="mb-dot"></div>Online · RAG Active</div>
</div>
"""

STATS_HTML = """
<div class="mb-stats">
  <div class="mb-sc"><div class="mb-sv">383</div><div class="mb-sl">Stores</div></div>
  <div class="mb-sc"><div class="mb-sv">253K+</div><div class="mb-sl">Medicines</div></div>
  <div class="mb-sc"><div class="mb-sv o">16K+</div><div class="mb-sl">Q&A Pairs</div></div>
  <div class="mb-sc"><div class="mb-sv g">40</div><div class="mb-sl">Areas</div></div>
  <div class="mb-sc"><div class="mb-sv w">100%</div><div class="mb-sl">Offline</div></div>
</div>
"""

PANEL_TOP = """
<div id="mb-panel">
  <div class="mb-sh">⚡ Quick Questions</div>
"""

PANEL_BOT = """
  <div class="mb-sd"></div>
  <div class="mb-ss">📍 Top Areas</div>
  <div class="mb-tags">
    <span class="mb-tag">Malviya Nagar</span>
    <span class="mb-tag">Vaishali Nagar</span>
    <span class="mb-tag">C-Scheme</span>
    <span class="mb-tag">Mansarovar</span>
    <span class="mb-tag">Tonk Road</span>
    <span class="mb-tag">Sanganer</span>
    <span class="mb-tag">Jagatpura</span>
    <span class="mb-tag">Ajmer Road</span>
    <span class="mb-tag">Raja Park</span>
    <span class="mb-tag">Bani Park</span>
  </div>
  <div class="mb-sd"></div>
  <div class="mb-ss">🔧 Tech Stack</div>
  <div class="mb-tr"><span class="mb-tk">LLM</span><span class="mb-tv">Mistral 7B</span></div>
  <div class="mb-tr"><span class="mb-tk">Embeddings</span><span class="mb-tv">MiniLM-L6</span></div>
  <div class="mb-tr"><span class="mb-tk">Vector DB</span><span class="mb-tv">ChromaDB</span></div>
  <div class="mb-tr"><span class="mb-tk">Location</span><span class="mb-tv">GeoPy</span></div>
  <div class="mb-tr"><span class="mb-tk">UI</span><span class="mb-tv">Gradio</span></div>
</div>
"""


# ══════════════════════════════════════════════════════════════════════════════
# GRADIO UI
# ══════════════════════════════════════════════════════════════════════════════
with gr.Blocks(css=css, title="MediBot Jaipur") as demo:

    gr.HTML(FONTS_HTML)
    gr.HTML(HDR_HTML)
    gr.HTML(STATS_HTML)

    with gr.Row(equal_height=True):

        # ── LEFT: Chat column ──
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                value=[{"role": "assistant", "content": "👋 Hello! I'm **MediBot Jaipur** — your local pharmacy assistant. Ask me about medicines, dosages, prices, or nearby pharmacies in Jaipur."}],
                height=560,
                show_label=False,
                elem_id="mb-bot",
            )
            area_box = gr.Textbox(
                placeholder="Your locality in Jaipur (e.g. Malviya Nagar, C-Scheme ...)",
                label="Location",
                lines=1,
                elem_id="mb-loc",
            )
            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="Ask about any medicine — price, composition, dosage, side effects ...",
                    label="",
                    scale=5,
                    container=False,
                    lines=1,
                    elem_id="mb-q",
                )
                submit_btn = gr.Button("Send ↑", variant="primary", scale=1, elem_id="mb-send")
            clear_btn = gr.Button("Clear conversation", size="sm", elem_id="mb-clear")

        # ── RIGHT: Sidebar ──
        with gr.Column(scale=1, min_width=270):
            gr.HTML(PANEL_TOP)
            q1 = gr.Button("Paracetamol — uses & dosage",       size="sm", elem_id="mb-q1")
            q2 = gr.Button("Price of Augmentin 625 tablet",     size="sm", elem_id="mb-q2")
            q3 = gr.Button("Common medicines for fever",         size="sm", elem_id="mb-q3")
            q4 = gr.Button("Azithromycin side effects",          size="sm", elem_id="mb-q4")
            q5 = gr.Button("Medicines with Amoxicillin",         size="sm", elem_id="mb-q5")
            q6 = gr.Button("Safe sleep-aid medicines",           size="sm", elem_id="mb-q6")
            gr.HTML(PANEL_BOT)

    # ── EVENT BINDINGS ──
    submit_btn.click(
        ask_assistant,
        inputs=[question_box, area_box, chatbot],
        outputs=[chatbot, question_box],
    )
    question_box.submit(
        ask_assistant,
        inputs=[question_box, area_box, chatbot],
        outputs=[chatbot, question_box],
    )
    clear_btn.click(clear_chat, outputs=[chatbot, question_box])

    q1.click(lambda a, h: ask_assistant("What is Paracetamol used for and what is the dosage?", a, h), inputs=[area_box, chatbot], outputs=[chatbot, question_box])
    q2.click(lambda a, h: ask_assistant("What is the price of Augmentin 625?", a, h),             inputs=[area_box, chatbot], outputs=[chatbot, question_box])
    q3.click(lambda a, h: ask_assistant("What medicines are commonly used for fever?", a, h),     inputs=[area_box, chatbot], outputs=[chatbot, question_box])
    q4.click(lambda a, h: ask_assistant("What are the side effects of Azithromycin?", a, h),      inputs=[area_box, chatbot], outputs=[chatbot, question_box])
    q5.click(lambda a, h: ask_assistant("What medicines contain Amoxicillin?", a, h),             inputs=[area_box, chatbot], outputs=[chatbot, question_box])
    q6.click(lambda a, h: ask_assistant("What are safe medicines for sleep?", a, h),              inputs=[area_box, chatbot], outputs=[chatbot, question_box])


# ══════════════════════════════════════════════════════════════════════════════
# LAUNCH
# ══════════════════════════════════════════════════════════════════════════════
demo.launch(
    share=False,          # set True to get a public gradio.live link
    inbrowser=True,       # auto-opens browser tab
    show_error=True,
)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
